# ECG Classification - PyTorch CNN Model (v2-pytorch)**What's New:**✅ PyTorch Implementation (from TensorFlow)✅ Fixed Data Leakage (scaler fits only on training data)✅ Stronger Regularization (dropout 0.3→0.5, weight decay)✅ Direct ONNX Export

## STEP 1: Import Libraries

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltfrom sklearn.preprocessing import StandardScalerfrom sklearn.model_selection import train_test_splitfrom sklearn.metrics import confusion_matrix, classification_report, accuracy_scoreimport warningswarnings.filterwarnings('ignore')import torchimport torch.nn as nnimport torch.optim as optimfrom torch.utils.data import Dataset, DataLoaderprint(f'PyTorch version: {torch.__version__}')device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')print(f'Using device: {device}')RANDOM_STATE = 42torch.manual_seed(RANDOM_STATE)np.random.seed(RANDOM_STATE)

## STEP 2: Load Data

In [ ]:
df = pd.read_csv('../../dataset_aritmia_NEW.csv')print(f'Dataset shape: {df.shape}')print(f'\nLabel distribution:')print(df['label'].value_counts())

## STEP 3: Data Preprocessing - **FIXING DATA LEAKAGE**

In [ ]:
X = df.drop('label', axis=1).valuesy = df['label'].values# STEP 3.1: SPLIT FIRST (prevents data leakage)X_train, X_temp, y_train, y_temp = train_test_split(    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)X_val, X_test, y_val, y_test = train_test_split(    X_temp, y_temp, test_size=0.5, random_state=RANDOM_STATE, stratify=y_temp)print(f'Training: {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)')print(f'Validation: {X_val.shape[0]} samples ({X_val.shape[0]/len(X)*100:.1f}%)')print(f'Test: {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.1f}%)')# STEP 3.2: NORMALIZE - Fit ONLY on training datascaler = StandardScaler()X_train_normalized = scaler.fit_transform(X_train)X_val_normalized = scaler.transform(X_val)X_test_normalized = scaler.transform(X_test)print(f'\nNormalization (training data only)')# Reshape for CNN (samples, channels, length)X_train_reshaped = X_train_normalized.reshape(-1, 1, 188)X_val_reshaped = X_val_normalized.reshape(-1, 1, 188)X_test_reshaped = X_test_normalized.reshape(-1, 1, 188)

## STEP 4: PyTorch Dataset and DataLoader

In [ ]:
class ECGDataset(Dataset):    def __init__(self, X, y):        self.X = torch.FloatTensor(X)        self.y = torch.LongTensor(y)    def __len__(self):        return len(self.X)    def __getitem__(self, idx):        return self.X[idx], self.y[idx]train_dataset = ECGDataset(X_train_reshaped, y_train)val_dataset = ECGDataset(X_val_reshaped, y_val)test_dataset = ECGDataset(X_test_reshaped, y_test)BATCH_SIZE = 32train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)print(f'DataLoaders created: {len(train_loader)} training batches')

## STEP 5: PyTorch CNN Model**Regularization:** Dropout 0.5, BatchNorm, Weight Decay

In [ ]:
class ECG_CNN(nn.Module):    def __init__(self, num_classes=2, dropout=0.5):        super(ECG_CNN, self).__init__()                # Block 1        self.conv1 = nn.Conv1d(1, 32, kernel_size=5, padding=2)        self.bn1 = nn.BatchNorm1d(32)        self.conv2 = nn.Conv1d(32, 32, kernel_size=5, padding=2)        self.bn2 = nn.BatchNorm1d(32)        self.pool1 = nn.MaxPool1d(2)        self.dropout1 = nn.Dropout(dropout)                # Block 2        self.conv3 = nn.Conv1d(32, 64, kernel_size=5, padding=2)        self.bn3 = nn.BatchNorm1d(64)        self.conv4 = nn.Conv1d(64, 64, kernel_size=5, padding=2)        self.bn4 = nn.BatchNorm1d(64)        self.pool2 = nn.MaxPool1d(2)        self.dropout2 = nn.Dropout(dropout)                # Block 3        self.conv5 = nn.Conv1d(64, 128, kernel_size=5, padding=2)        self.bn5 = nn.BatchNorm1d(128)        self.conv6 = nn.Conv1d(128, 128, kernel_size=5, padding=2)        self.bn6 = nn.BatchNorm1d(128)        self.pool3 = nn.MaxPool1d(2)        self.dropout3 = nn.Dropout(dropout)                # Block 4        self.conv7 = nn.Conv1d(128, 256, kernel_size=5, padding=2)        self.bn7 = nn.BatchNorm1d(256)        self.pool4 = nn.AdaptiveAvgPool1d(1)                # FC layers        self.fc1 = nn.Linear(256, 128)        self.bn_fc1 = nn.BatchNorm1d(128)        self.dropout_fc1 = nn.Dropout(dropout)        self.fc2 = nn.Linear(128, 64)        self.bn_fc2 = nn.BatchNorm1d(64)        self.dropout_fc2 = nn.Dropout(dropout)        self.fc3 = nn.Linear(64, num_classes)                self.relu = nn.ReLU()        def forward(self, x):        # Block 1        x = self.relu(self.bn1(self.conv1(x)))        x = self.relu(self.bn2(self.conv2(x)))        x = self.pool1(x)        x = self.dropout1(x)                # Block 2        x = self.relu(self.bn3(self.conv3(x)))        x = self.relu(self.bn4(self.conv4(x)))        x = self.pool2(x)        x = self.dropout2(x)                # Block 3        x = self.relu(self.bn5(self.conv5(x)))        x = self.relu(self.bn6(self.conv6(x)))        x = self.pool3(x)        x = self.dropout3(x)                # Block 4        x = self.relu(self.bn7(self.conv7(x)))        x = self.pool4(x)        x = x.view(x.size(0), -1)                # FC        x = self.relu(self.bn_fc1(self.fc1(x)))        x = self.dropout_fc1(x)        x = self.relu(self.bn_fc2(self.fc2(x)))        x = self.dropout_fc2(x)        x = self.fc3(x)                return xmodel = ECG_CNN(num_classes=2, dropout=0.5).to(device)print(f'Total parameters: {sum(p.numel() for p in model.parameters())}')

## STEP 6: Training Configuration

In [ ]:
from sklearn.utils.class_weight import compute_class_weightclass_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)class_weights = torch.FloatTensor(class_weights).to(device)criterion = nn.CrossEntropyLoss(weight=class_weights)optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)print('Training configured')

## STEP 7: Training Functions

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):    model.train()    running_loss, correct, total = 0.0, 0, 0    for inputs, labels in loader:        inputs, labels = inputs.to(device), labels.to(device)        optimizer.zero_grad()        outputs = model(inputs)        loss = criterion(outputs, labels)        loss.backward()        optimizer.step()        running_loss += loss.item()        _, predicted = outputs.max(1)        total += labels.size(0)        correct += predicted.eq(labels).sum().item()    return running_loss / len(loader), correct / totaldef validate_epoch(model, loader, criterion, device):    model.eval()    running_loss, correct, total = 0.0, 0, 0    with torch.no_grad():        for inputs, labels in loader:            inputs, labels = inputs.to(device), labels.to(device)            outputs = model(inputs)            loss = criterion(outputs, labels)            running_loss += loss.item()            _, predicted = outputs.max(1)            total += labels.size(0)            correct += predicted.eq(labels).sum().item()    return running_loss / len(loader), correct / totalprint('Training functions ready')

## STEP 8: Training Loop

In [ ]:
NUM_EPOCHS, PATIENCE = 100, 10best_val_loss, patience_counter = float('inf'), 0train_losses, val_losses, train_accs, val_accs = [], [], [], []print('Starting training...')for epoch in range(NUM_EPOCHS):    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)    val_loss, val_acc = validate_epoch(model, val_loader, criterion, device)        train_losses.append(train_loss)    val_losses.append(val_loss)    train_accs.append(train_acc)    val_accs.append(val_acc)        scheduler.step(val_loss)        if (epoch + 1) % 5 == 0 or epoch < 5:        print(f'Epoch [{epoch+1}/{NUM_EPOCHS}] Train: {train_acc:.4f} Val: {val_acc:.4f}')        if val_loss < best_val_loss:        best_val_loss = val_loss        patience_counter = 0        torch.save({'model_state_dict': model.state_dict(), 'epoch': epoch, 'val_acc': val_acc}, 'ecg_cnn_pytorch_best.pth')    else:        patience_counter += 1        if patience_counter >= PATIENCE:            print(f'Early stopping at epoch {epoch+1}')            breakcheckpoint = torch.load('ecg_cnn_pytorch_best.pth')model.load_state_dict(checkpoint['model_state_dict'])print(f'\nBest model from epoch {checkpoint["epoch"]+1}, val_acc: {checkpoint["val_acc"]:.4f}')

## STEP 9: Evaluation

In [ ]:
test_loss, test_acc = validate_epoch(model, test_loader, criterion, device)print(f'Test Accuracy: {test_acc:.4f}')# Get predictionsmodel.eval()all_preds, all_labels = [], []with torch.no_grad():    for inputs, labels in test_loader:        outputs = model(inputs.to(device))        _, predicted = outputs.max(1)        all_preds.extend(predicted.cpu().numpy())        all_labels.extend(labels.numpy())print('\nClassification Report:')print(classification_report(all_labels, all_preds, target_names=['Normal', 'Abnormal']))

## STEP 10: Save Models

In [ ]:
torch.save({'model_state_dict': model.state_dict(), 'test_acc': test_acc}, 'ecg_cnn_v2_pytorch_final.pth')import joblibjoblib.dump(scaler, 'scaler_v2_pytorch.pkl')print('Models saved')

## STEP 11: Export to ONNX

In [ ]:
try:    model.eval()    dummy_input = torch.randn(1, 1, 188).to(device)    torch.onnx.export(model, dummy_input, 'ecg_cnn_v2_pytorch_final.onnx',                     export_params=True, opset_version=13,                     input_names=['input'], output_names=['output'],                     dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}})    print('✓ ONNX model exported: ecg_cnn_v2_pytorch_final.onnx')        import onnxruntime as ort    ort_session = ort.InferenceSession('ecg_cnn_v2_pytorch_final.onnx')    print('✓ ONNX model verified')except Exception as e:    print(f'ONNX export failed: {e}')